# Réseau de neurones sur MNIST avec TensorFlow/Keras
## Version plus pédagogique et plus complète

Ce notebook explique chaque étape du projet : chargement des données, prétraitement, construction du modèle, entraînement, évaluation et deux exemples de prédiction.
L'objectif est de montrer non seulement le code, mais aussi le rôle de chaque bloc.

In [ ]:
%pip install tensorflow

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# Import des bibliothèques nécessaires
# numpy : manipuler les tableaux
# matplotlib : afficher les images et les courbes
# tensorflow.keras : construire et entraîner le réseau de neurones

# On fixe une graine pour rendre les résultats plus reproductibles.
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Chargement du jeu de données MNIST.
# x_train et x_test contiennent les images, y_train et y_test contiennent les labels.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print('Taille du jeu d\'entraînement :', x_train.shape)
print('Taille du jeu de test :', x_test.shape)
print('Exemple de label :', y_train[0])

In [ ]:
# On affiche plusieurs images pour comprendre à quoi ressemble la base de données.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
fig.suptitle('Quelques images MNIST')

for index, ax in enumerate(axes.flat):
    ax.imshow(x_train[index], cmap='gray')
    ax.set_title(f'Classe : {y_train[index]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# La normalisation transforme les pixels de l'intervalle 0..255 vers 0..1.
# Cela aide le réseau à apprendre plus facilement.
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print('Valeur minimale après normalisation :', x_train.min())
print('Valeur maximale après normalisation :', x_train.max())

In [ ]:
# Chaque image 28 x 28 doit être transformée en vecteur de 784 valeurs.
# On garde des noms explicites pour ne pas écraser les données d'origine.
x_train_flat = x_train.reshape(-1, 28 * 28)
x_test_flat = x_test.reshape(-1, 28 * 28)

print('Forme des images d\'entraînement après aplatissement :', x_train_flat.shape)
print('Forme des images de test après aplatissement :', x_test_flat.shape)

In [ ]:
# Définition du réseau de neurones.
# Input : 784 valeurs par image.
# Dense(128) puis Dense(64) : extraction progressive d'information.
# Dropout : aide à limiter le surapprentissage.
# Dense(10) avec softmax : retourne une probabilité pour chaque chiffre.
model = models.Sequential([
    layers.Input(shape=(28 * 28,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()

In [ ]:
# Compilation du modèle.
# adam est un optimiseur efficace pour ce type de problème.
# sparse_categorical_crossentropy est adapté aux labels codés comme des entiers.
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Entraînement avec une validation interne.
# validation_split=0.1 garde 10 % des données d'entraînement pour vérifier la généralisation.
history = model.fit(
    x_train_flat,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=2
)

In [ ]:
# Visualisation de l'apprentissage.
# Si les courbes d'entraînement et de validation restent proches, le modèle généralise mieux.
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Accuracy entraînement')
plt.plot(history.history['val_accuracy'], label='Accuracy validation')
plt.title('Précision')
plt.xlabel('Époque')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Loss entraînement')
plt.plot(history.history['val_loss'], label='Loss validation')
plt.title('Perte')
plt.xlabel('Époque')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Évaluation finale sur le jeu de test.
# Cette étape mesure le comportement du modèle sur des données nouvelles.
test_loss, test_accuracy = model.evaluate(x_test_flat, y_test, verbose=0)

print(f'Perte sur le jeu de test : {test_loss:.4f}')
print(f'Accuracy sur le jeu de test : {test_accuracy:.4f}')

In [ ]:
# Premier exemple de prédiction.
# On récupère une seule image sous la forme attendue par le modèle : (1, 784).
def show_prediction(index):
    image = x_test_flat[index:index + 1]
    probabilities = model.predict(image, verbose=0)[0]
    predicted_label = int(np.argmax(probabilities))
    true_label = int(y_test[index])

    plt.figure(figsize=(4, 4))
    plt.imshow(x_test[index], cmap='gray')
    plt.title(f'Vrai : {true_label} | Prédit : {predicted_label}')
    plt.axis('off')
    plt.show()

    print('Probabilités par classe :')
    for digit, probability in enumerate(probabilities):
        print(f'  {digit} -> {probability:.3f}')

show_prediction(0)

# Deuxième exemple : vérifier plusieurs images pour confirmer le comportement du modèle.
sample_indices = [1, 2, 3, 4, 5]
predicted_labels = []

for index in sample_indices:
    probabilities = model.predict(x_test_flat[index:index + 1], verbose=0)[0]
    predicted_label = int(np.argmax(probabilities))
    true_label = int(y_test[index])
    predicted_labels.append(predicted_label)
    print(f'Image {index} -> vrai : {true_label} | prédit : {predicted_label}')

print('Liste des prédictions :', predicted_labels)